# MHC-Coach Test

This notebook tests only the code path used for `SriyaM/MHC-Coach` in `nudge-generation`.

- Provider path: Hugging Face (non-MLX)
- Runtime path: `transformers` + `torch`
- Goal: confirm model load + generation succeeds in Colab


In [ ]:
# Install runtime deps for Colab
!pip -q install -U transformers accelerate sentencepiece safetensors

In [ ]:
import json
from typing import Any, Optional

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "SriyaM/MHC-Coach"

In [ ]:
def resolve_torch_device() -> str:
    # Mirrors python_service/huggingface_service.py
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def build_transformers_prompt(tokenizer: Any, prompt: str, device: str):
    # Mirrors python_service/huggingface_service.py
    if hasattr(tokenizer, "apply_chat_template"):
        messages = [{"role": "user", "content": prompt}]
        tokenized = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt",
        )
    else:
        tokenized = tokenizer(prompt, return_tensors="pt")

    if isinstance(tokenized, dict):
        moved = {key: value.to(device) for key, value in tokenized.items()}
        prompt_tokens = moved["input_ids"].shape[-1]
        return moved, prompt_tokens

    moved = tokenized.to(device)
    prompt_tokens = moved.shape[-1]
    return {"input_ids": moved}, prompt_tokens


def generate_transformers_text(
    model,
    tokenizer,
    prompt: str,
    max_tokens: int = 512,
    temperature: Optional[float] = 0.7,
):
    # Mirrors non-MLX branch in python_service/huggingface_service.py
    device = resolve_torch_device()
    inputs, prompt_tokens = build_transformers_prompt(tokenizer, prompt, device)

    do_sample = temperature is not None and temperature > 0
    eos_token_id = tokenizer.eos_token_id
    pad_token_id = tokenizer.pad_token_id
    resolved_pad_token_id = (
        eos_token_id
        if eos_token_id is not None
        else pad_token_id
        if pad_token_id is not None
        else 0
    )

    generation_kwargs = {
        "max_new_tokens": max_tokens,
        "do_sample": do_sample,
        "pad_token_id": resolved_pad_token_id,
    }
    if do_sample:
        generation_kwargs["temperature"] = float(temperature)

    with torch.no_grad():
        output_ids = model.generate(**inputs, **generation_kwargs)

    completion_ids = output_ids[0][prompt_tokens:]
    decoded = tokenizer.decode(completion_ids, skip_special_tokens=True)
    completion = "".join(decoded).strip() if isinstance(decoded, list) else str(decoded).strip()
    return completion


In [ ]:
device = resolve_torch_device()
dtype = torch.float16 if device in {"cuda", "mps"} else torch.float32
print(f"Using device: {device}, dtype: {dtype}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=dtype)
model.to(device)
model.eval()
print("Model and tokenizer loaded")

In [ ]:
# Build a single prompt in the same style as src/generateNudgePermutations.ts
language_context = ""
gender_context = "This participant is female."
age_context = "This participant is 42 years old and in addition to thinking about the short-term benefits of exercise on their mood, energy, and health, should also now be thinking about their long-term risk of chronic disease development, such as cardiovascular disease, dementia, and cancer, unless they already have a chromic condition. IF THEY ALREADY HAVE A CHRONIC CONDITION, they should be thinking about inproving quality of life through exercise."
disease_context = "This participant has diabetes, a condition that is characterized by high glucose levels and insulin resistance. Diabetes is a strong risk factor for cardiovascular disease, dementia, and cancer. Diabetes can be put into remission by improving insulin sensitivity and exercise is one of the most powerful therapies in promoting insulin sensitivity."
stage_context = "This person is in the preparation stage of changing their exercise habits. This person is ready to begin exercising in the next 30 days and has begun taking small steps."
education_context = "This person is more highly educated and has some form of higher education. Please write the prompts at the 12th grade reading comprehension level."
notification_time_context = "This user prefers to receive recommendation at 7:00 AM. Tailor the prompt to match the typical context of that time of day and suggest realistic opportunities for activity they could do the same day they recieve the prompt, even if it is late evening. For instance, if the time is in the morning, encourage early activity or planning for later (e.g., lunch or after work). Avoid irrelevant examples that do not fit the selected time of day."

prompt = (
    "Write 7 motivational messages that are proper length to go in a push notification using a calm, encouraging, and professional tone, like that of a health coach to motivate a smartphone user in increase physical activity levels.  Also create a title for each of push notifications that is a short summary/call to action of the push notification that is paired with it. Return the response as a JSON array with exactly 7 objects, each having \"title\" and \"body\" fields. If there is a disease context given, you can reference that disease in some of the nudges. When generating nudges, avoid the word 'healthy' and remove unnecessary qualifiers such as 'brisk' or 'deep'. Suggest only simple, low-risk activities without adding extra exercises or medical disclaimers not provided. Keep messages concise, calm, and practical; focus on one clear activity with plain language. Keep recommendations practical, varied, and easy to integrate into daily routines. NEVER USE EM DASHES, EMOJIS OR ABBREVIATIONS FOR DISEASES IN THE NUDGE. Each nudge should be personalized to the following information: "
    f"{language_context} {gender_context} {age_context} {disease_context} {stage_context} {education_context} {notification_time_context} "
    "Think carefully before delivering the prompts to ensure they are personalized to the information given (especially any given disease context) and give recommendations based on research backed motivational methods."
)

response_text = generate_transformers_text(
    model=model,
    tokenizer=tokenizer,
    prompt=prompt,
    max_tokens=512,
    temperature=0.7,
)

print(response_text[:2000])
print("\n--- response length:", len(response_text))

In [ ]:
# Optional: quick JSON validity check (best effort)
try:
    parsed = json.loads(response_text)
    print("Valid JSON:", isinstance(parsed, list), "items:", len(parsed) if isinstance(parsed, list) else "n/a")
except Exception as exc:
    print("Not valid JSON:", exc)